# Validação do ambiente U-Mamba (estágio 11)

Prepara o Google Colab para executar a arquitetura oficial `UMambaEnc_2d` sem compilar o Mamba do zero.

**Antes de executar:** selecione uma GPU NVIDIA (T4 ou outra disponível).

**Importante:** na primeira execução, o notebook pode instalar um stack binário compatível e reiniciar o runtime automaticamente. Quando o Colab reconectar, execute **tudo novamente uma segunda vez**.

## 1. Bootstrap do TCC

Carrega o código mais recente do repositório.

In [1]:
import importlib
import pathlib
import sys
import urllib.request

BOOTSTRAP_URL = "https://raw.githubusercontent.com/oguel/tcc-umamba/main/src/bootstrap.py"
pathlib.Path("bootstrap.py").write_bytes(urllib.request.urlopen(BOOTSTRAP_URL).read())
sys.path.insert(0, str(pathlib.Path.cwd()))
bootstrap = importlib.import_module("bootstrap")
importlib.reload(bootstrap)
workspace = bootstrap.bootstrap_workspace()
print(f"Workspace: {workspace}")


Workspace: /content


## 2. Preparar stack binário do Mamba

Usa PyTorch 2.10 + CUDA 12.6 e uma wheel CUDA pré-compilada do `mamba-ssm`. Isso evita a compilação local que pode levar dezenas de minutos no Colab.

In [2]:
from src.models.umamba_runtime import install_prebuilt_colab_stack, restart_colab_runtime

restart_required = install_prebuilt_colab_stack()
if restart_required:
    restart_colab_runtime()
else:
    print("Stack binário já está compatível. Nenhum reinício necessário.")


PyTorch compatível já instalado: 2.10.0+cu126
mamba-ssm já instalado: 2.3.2.post1
Stack binário já está compatível. Nenhum reinício necessário.


## 3. Confirmar GPU e versões

Depois do reinício, confirme que CUDA e o stack fixado estão ativos.

In [3]:
import platform
import torch

print(f"Sistema: {platform.platform()}")
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA do PyTorch: {torch.version.cuda}")
print(f"CUDA disponível: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("GPU CUDA não está ativa no Colab.")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")


Sistema: Linux-6.6.122+-x86_64-with-glibc2.39
Python: 3.13.15
PyTorch: 2.10.0+cu126
CUDA do PyTorch: 12.6
CUDA disponível: True
GPU: Tesla T4
VRAM total: 14.56 GB


## 4. Validar Mamba e preparar U-Mamba oficial

Executa um bloco Mamba na GPU e prepara a arquitetura `UMambaEnc_2d` oficial em um commit fixado.

In [4]:
import json
from src.models.umamba_runtime import ensure_umamba_runtime

runtime_info = ensure_umamba_runtime()
print(json.dumps(runtime_info, indent=2, ensure_ascii=False))


{
  "python": "3.13.15",
  "torch": "2.10.0+cu126",
  "torch_cuda": "12.6",
  "torch_cxx11_abi": true,
  "gpu": "Tesla T4",
  "mamba_ssm": "2.3.2.post1",
  "umamba_commit": "28459e33ca03769800dd35e23c6e62491d1925b5",
  "architecture_file": "/content/umamba_runtime/umamba_official_standalone.py"
}


/content/umamba_runtime/umamba_official_standalone.py:87: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled=False)


## 5. Smoke test de treinamento U-Mamba RGB

Constrói `UMambaEnc_2d` com entrada RGB 256×256 e executa um passo completo de treinamento na GPU: forward, BCE+Dice, backward e AdamW. Isso valida tanto inferência quanto os kernels CUDA usados para gradientes antes do notebook 12.

In [5]:
from torch.optim import AdamW

from src.config import get_config
from src.losses import BCEDiceLoss
from src.models.umamba import build_official_umamba_enc_2d

config = get_config()
features = tuple(int(v) for v in config["model"]["umamba_features"])
device = torch.device("cuda")

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model = build_official_umamba_enc_2d(
    input_channels=3,
    num_classes=1,
    input_size=(256, 256),
    features_per_stage=features,
).to(device)
model.train()

x = torch.randn(1, 3, 256, 256, device=device)
target = (torch.rand(1, 1, 256, 256, device=device) > 0.7).float()
criterion = BCEDiceLoss(bce_weight=1.0, dice_weight=1.0)
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

optimizer.zero_grad(set_to_none=True)
y = model(x)
loss = criterion(y, target)

if not torch.isfinite(loss):
    raise RuntimeError(f"Loss não finita no smoke test: {loss.item()}")

loss.backward()

gradients = [p.grad for p in model.parameters() if p.grad is not None]
if not gradients:
    raise RuntimeError("Nenhum gradiente foi produzido pela U-Mamba.")
if not all(torch.isfinite(g).all().item() for g in gradients):
    raise RuntimeError("Gradientes não finitos detectados no smoke test U-Mamba.")

optimizer.step()
torch.cuda.synchronize()

peak_vram_mb = torch.cuda.max_memory_allocated() / 1024**2
parameters = sum(p.numel() for p in model.parameters())

print(f"Entrada: {tuple(x.shape)}")
print(f"Saída: {tuple(y.shape)}")
print(f"Loss smoke test: {loss.item():.6f}")
print(f"Parâmetros: {parameters:,}")
print(f"Pico de VRAM (forward + backward): {peak_vram_mb:.1f} MB")

if tuple(y.shape) != (1, 1, 256, 256):
    raise RuntimeError(f"Shape inesperado: {tuple(y.shape)}")

print("SMOKE TEST U-MAMBA (FORWARD + BACKWARD + OPTIMIZER): OK")


feature_map_sizes: [[256, 256], [128, 128], [64, 64], [32, 32]]
do_channel_token: [False, False, False, False]
MambaLayer: dim: 32
MambaLayer: dim: 128
Entrada: (1, 3, 256, 256)
Saída: (1, 1, 256, 256)
Loss smoke test: 1.702868
Parâmetros: 723,091
Pico de VRAM (forward + backward): 223.3 MB
SMOKE TEST U-MAMBA (FORWARD + BACKWARD + OPTIMIZER): OK


## 6. Salvar relatório no Drive


In [6]:
from src import io

storage_paths = io.resolve_storage_paths()
runs_dir = storage_paths["artifacts_runs"]
runs_dir.mkdir(parents=True, exist_ok=True)
report = {
    **runtime_info,
    "input_shape": list(x.shape),
    "output_shape": list(y.shape),
    "parameters": parameters,
    "smoke_test_peak_vram_mb": peak_vram_mb,
    "smoke_test_loss": float(loss.detach().item()),
    "training_step_validated": True,
    "status": "ok",
}
report_path = runs_dir / "umamba_environment.json"
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Relatório salvo em: {report_path}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Relatório salvo em: /content/drive/MyDrive/tcc/artifacts/runs/umamba_environment.json


## Critério para seguir ao notebook 12

Só prossiga quando aparecer **`SMOKE TEST U-MAMBA (FORWARD + BACKWARD + OPTIMIZER): OK`**. Nesse ponto já terão sido validados construção da arquitetura, forward, loss, backward e atualização de pesos na GPU.